In [ ]:
#@title Install pacakges


!pip install rasterio
!pip install geojson

!pip install cairosvg
!pip install elapid

# Set up

In [ ]:
#@title Load pacakges
import os
import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go

import geopandas as gpd
from shapely.geometry import Point
import geojson
from pprint import pprint
import rasterio
from rasterio.mask import mask
import rasterio.plot as rioplot
import elapid as ela
from sklearn import metrics
from glob import glob
import time

import geemap.core as geemap
from datetime import datetime
import folium
from folium.features import GeoJsonTooltip
from folium.features import DivIcon

import cairosvg
from PIL import Image
import io

import pickle

# preferences
%matplotlib inline
mpl.style.use('ggplot')

In [ ]:
print(f"Notebook last run with elapid version {ela.__version__}")
print(f"Notebook last run with numpy version {np.__version__}")
print(f"Notebook last run with folium version {folium.__version__}")
print(f"Notebook last run with rasterio version {rasterio.__version__}")

In [ ]:
#@title Set directory

from google.colab import drive
drive.mount('/content/drive/')
print(sorted(os.listdir()))
try:
  os.chdir('/content/.....') #set to your directory path
  print(sorted(os.listdir()))
except:
  print("GGDrive may not have been updated, wait and retry")

In [ ]:
#@title Add system path for scripts

import sys

if 'scripts' not in sys.path:
    sys.path.append('scripts')
print(sys.path)

In [ ]:
#@title import from scripts

import importlib

import preprocess_fawdata as pf
importlib.reload(pf)
import visualisation_fawdata as vf
importlib.reload(vf)
import preprocess_climatedata as pc
importlib.reload(pc)
import visualisation_climatedata as vc
importlib.reload(vc)
import augment_elapid as ae
importlib.reload(ae)
import visualisation_elapid as ve
importlib.reload(ve)

# Load input choice

## Landscape

In [ ]:
#@title Landscape
pnv_vecea_excludeWDDa_dissolved_extend_file = "data/spatial_data_publication/pnv_vecea_excludeWDDa_dissolved_extend.geojson"
pnv_vecea_excludeWDDa_dissolved_extend_gdf = gpd.read_file(pnv_vecea_excludeWDDa_dissolved_extend_file)
landscape_gdf = pnv_vecea_excludeWDDa_dissolved_extend_gdf
landscape_gdf = landscape_gdf.to_crs(epsg=4326)
augment_cleaned_counties_gdf = gpd.read_file("data/spatial_data_publication/augment_cleaned_counties_gdf.geojson")
kenya_amd1_gdf = gpd.read_file("data/spatial_data_publication/gadm41_KEN_1.json")
kenya_amd1_gdf = kenya_amd1_gdf.to_crs(epsg=4326)
kenya_amd2_gdf = gpd.read_file("data/spatial_data_publication/gadm41_KEN_2.json")
kenya_amd2_gdf = kenya_amd2_gdf.to_crs(epsg=4326)

## Presence samples

In [ ]:
#@title late_df for temporal transferability
late_df = pd.read_csv('1_data/faw_data/processed/late_df_gdf.csv')
late_df_gdf = gpd.GeoDataFrame(late_df[["species", "Date", "search_state"]],
                                          geometry=gpd.points_from_xy(late_df.lon, late_df.lat), crs="EPSG:4326")
late_df_gdf

In [ ]:
#@title faw_processed_formaxent_publication

input_pres_dir = '1_data/faw_data/processed'
preschoice_dict = {}
for filename in glob(os.path.join(input_pres_dir, '*.geojson')):
    name = os.path.splitext(os.path.basename(filename))[0]
    name = name.replace('early_', '').replace('_partition', '').replace('_','')
    print(f"Loading {name} from {filename}")
    try:
        preschoice_dict[name] = gpd.read_file(filename)
        print(f"Loaded {name} from {filename}")
        print(preschoice_dict[name].head(3))
        print(preschoice_dict[name].shape)
    except Exception as e:
        print(f"Error loading {filename}: {e}")

In [ ]:
thinned_list = [value for key,value in list(preschoice_dict.items())[1:]]
colormap = ["blue", "green", "purple"]
# list(bgchoice_dict.items())[:2])
labels = [key for key,value in list(preschoice_dict.items())[1:]]
fig, axes = plt.subplots(3, 1, figsize=(6, 15))
axes = axes.flatten()
for i in range(len(thinned_list)):
    ax = axes[i]
    augment_cleaned_counties_gdf.plot(ax=ax, color='gray', alpha=0.5, edgecolor = 'black')
    thinned_list[i].plot(ax=ax, marker='o', color=colormap[i], markersize=5, label=f"{labels[i]} km")
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(f'{labels[i]} km (n={len(thinned_list[i])})')
    ax.legend()
plt.tight_layout()

plt.show()

## Background samples

In [ ]:
input_bg_dir = '1_data/bg_generated_data'
bgchoice_dict = {}
for filename in glob(os.path.join(input_bg_dir, '*.geojson')):
    name = os.path.splitext(os.path.basename(filename))[0]
    print(f"Loading {name} from {filename}")
    try:
        bgchoice_dict[name] = gpd.read_file(filename)
        print(f"Loaded {name} from {filename}")
        print(bgchoice_dict[name].head(3))
        print(bgchoice_dict[name].shape)
    except Exception as e:
        print(f"Error loading {filename}: {e}")

In [ ]:
#@title all bg spatial plots

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for i in range(len(axes)):
  landscape_gdf.plot(ax=axes[i], color='gray', alpha=0.5)
  kenya_amd1_gdf.plot(ax=axes[i], edgecolor='white', color="none", linewidth=0.8)
  axes[i].set_xlabel('Longitude')
  axes[i].set_ylabel('Latitude')

bgchoice_dict['bg1uniform'].plot(ax=axes[0], marker='x', color='brown', markersize=1, label='bg1uniform')
early_df_gdf.plot(ax=axes[0], markersize=1, color='black', label='early_all')
axes[0].set_title('Uniform Background Sample')
axes[0].legend(fontsize=10, markerscale=5)

bgchoice_dict['bg2lai'].plot(ax=axes[1], marker='x', color='navy', markersize=1, label='bg2lai')
early_df_gdf.plot(ax=axes[1], markersize=1, color='black', label='early_all')
axes[1].set_title('Biased Background Sample\n(leaf_area_index_low_vegetation 2013-2024)')
axes[1].legend(fontsize=10, markerscale=5)

bgchoice_dict['bg3meanA3yr'].plot(ax=axes[2], marker='x', color='teal', markersize=1, label='bg3meanA3yr')
early_df_gdf.plot(ax=axes[2], markersize=1, color='black', label='early_all')
axes[2].set_title('Biased Background Sample\n(mean maize planting area 2020, 2022, 2023)')
axes[2].legend(fontsize=10, markerscale=5)

panel_labels = ["A", "B", "C"]
for i, ax in enumerate(axes):
    ax.text(
        0.02, 0.97, panel_labels[i],         # x, y (axes coords)
        transform=ax.transAxes,
        fontsize=16,
        # fontweight="bold",
        va="top", ha="left"
    )

plt.tight_layout()
plt.show()

## Climate data

In [ ]:
#@title Avg climate loaded
raster_dict_era5land = ae.load_climatetif_to_dict_rec(climate_folder_path = '1_data/climate_data/climate_processed_data/new_gee_avg_era5land_clipped')
raster_dict_bioclim = ae.load_climatetif_to_dict_rec(climate_folder_path = '1_data/climate_data/climate_processed_data/annual_longterm_wc2.1_30s_bio_clipped_setnodata')
raster_dict_monthly_bg = ae.load_climatetif_to_dict_rec(climate_folder_path = '1_data/climate_data/climate_processed_data/monthly_longterm_wc2.1_2.5min_clipped_allmonth_mean')

In [ ]:
#@title Time-specific climate loaded

raster_dict_monthly = ae.load_climatetif_to_dict_rec(climate_folder_path = '1_data/climate_data/climate_processed_data/monthly_longterm_wc2.1_2.5min_clipped_split_folder_setnodata')
raster_dict_yearmonthly = ae.load_climatetif_to_dict_rec(climate_folder_path = '1_data/climate_data/climate_processed_data/new_gee_selected_months_era5land_clipped_split_folder')
# Just for illustration - because at the points of using them (calling annotate_by_time()), we read data from their raw files, rather than from the dict we created (unlike Avg climate dict)

In [ ]:
#@title Clean labels as necessary
raster_dict_era5land.keys()

# raster_dict_era5land
new_labels = []
for label in raster_dict_era5land.keys():
  old_label = label
  new_label = old_label.replace('_raster_tif_clipped', '')
  new_labels.append(new_label)
raster_dict_era5land_relabelled = {
    new_label: raster_dict_era5land[old_label]
    for old_label, new_label in zip(raster_dict_era5land.keys(), new_labels)
}
raster_dict_era5land_relabelled
raster_dict_era5land_relabelled.pop('leaf_area_index_low_vegetation')


raster_dict_monthly_bg.keys()
# raster_dict_monthly_bg
new_labels = []
for label in raster_dict_monthly_bg.keys():
  old_label = label
  new_label = old_label.replace('_allmonth_mean', '')
  new_labels.append(new_label)
raster_dict_monthlybg_relabelled = {
    new_label: raster_dict_monthly_bg[old_label]
    for old_label, new_label in zip(raster_dict_monthly_bg.keys(), new_labels)
}
raster_dict_monthlybg_relabelled

# Prepare for running with 3 avg climate

Outputs (images and model list in .pkl) are in two subfolders: maxent_output_nov_19 (for thinned1km, thinned3km, thinned9km) or maxent_output_nov_18* (for thinned5km)
in a big folder  maxent_output_nov (in Publication folder)

To compare them side by side we will use the functions from visualise_elapid.py. *I just moved thinned5km's folders to maxent_output_nov_19, so it is easiest to compare using these functions I produced.

## check grid size choice

In [ ]:
# Check grid size choice
presence = preschoice_dict['thinned9km']
background = bgchoice_dict['bg1uniform']
rasters = [value for key, value in raster_dict_monthlybg_relabelled.items() if "leaf" not in key]
labels = [key for key, value in raster_dict_monthlybg_relabelled.items() if "leaf" not in key]
merged = ela.stack_geodataframes(presence, background, add_class_label=True)
annotated = ela.annotate(merged, rasters, labels=labels, drop_na=True, quiet=False)
kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
presence.crs
for gs in [0.2,0.3,0.4,0.5]:
  grid_size = gs
  train, test = ela.checkerboard_split(annotated, grid_size=grid_size)
  # re-merge them for plotting purposes
  train['split'] = 'train'
  test['split'] = 'test'
  checker = ela.stack_geodataframes(train, test)
  # plot test and train annotated merged samples in different colour
  ax = checker.plot(column='split', markersize=0.75, legend=True,figsize=(6,6))
  kmd_zone.plot(ax=ax, color='none', edgecolor='black')
  ax.set_xlabel('Longitude')
  ax.set_ylabel('Latitude')
  ax.set_title(f'Checkerboard split all samples\ngridsize = {gs} degree ({gs*110:.0f} km)')

# Prepare for running with time specific 2 climates 

m: monthly-value annotated presence + allmonth-value annotated background
ym: yearmonthly-value annotated presence + allyearmonth-value annotated background

## create monthly-match presence (anno_m_mergedpresthinned_cleaned)

In [ ]:
#@title create monthly-match presence (anno_m_mergedpresthinned_cleaned)
#1. create anno_m_mergedpresthinned
# using monthly_longterm_wc2.1_2.5min_clipped_split_folder_setnodata


# points_gdf are presence data - here we are using thinned 5 and 9 km
points_gdf = preschoice_dict['thinned9km']
date_column = 'Date'
raster_base_folder = 'data/climate_data/climate_processed_data/monthly_longterm_wc2.1_2.5min_clipped_split_folder_setnodata'
anno_m_mergedpresthinned = ae.annotate_points_by_time(points_gdf, date_column, raster_base_folder, date_format="%m")
display(anno_m_mergedpresthinned.head())

#2. clean the column name to be a base variable name
annotated_dfs = []
for time_key, group in anno_m_mergedpresthinned.groupby("time_key"):
    # Drop columns with any NA values in the current group
    group_cleaned = group.dropna(axis=1)
    # Clean column names
    cleaned_columns = {}
    for col in group_cleaned.columns:
        if col.startswith('wc2.1_30s_'):
            # print('From old col:',col)
            parts = col.split('_')
            # Remove 'wc2.1_30s_' and the last part (month number)
            new_col = '_'.join(parts[2:-1])
            print('From old col:',col,'To new col:',new_col)
            cleaned_columns[col] = new_col
        else:
            cleaned_columns[col] = col # Keep other columns as they are
    print(f'cleaned_columns for {time_key}', cleaned_columns)
    group_cleaned = group_cleaned.rename(columns=cleaned_columns)
    annotated_dfs.append(group_cleaned)
    print('current annotated_dfs length:', len(annotated_dfs))
# Concatenate the cleaned dataframes
anno_m_mergedpresthinned_cleaned = pd.concat(annotated_dfs, ignore_index=True)
# 3. add class to set it as presence points
anno_m_mergedpresthinned_cleaned['class'] = 1
display(anno_m_mergedpresthinned_cleaned.head())
display(anno_m_mergedpresthinned_cleaned.columns)

import pickle

with open("maxent_output_nov/maxent_output_nov_20/anno_m_9kmthinned_cleaned.pkl", "wb") as f:
    pickle.dump(anno_m_mergedpresthinned_cleaned, f)

# Do the same with 5km-thinned, 3km-thinned



## create yearmonthly-match presence (anno_ym_mergedpresthinned)

In [ ]:
#@title create yearmonthly-match presence (anno_ym_mergedpresthinned)
#1. create anno_ym_mergedpresthinned

# points_gdf are presence data - here we are using thinned 5 and 9 km
points_gdf = preschoice_dict['thinned3km']
date_column = 'Date'
# raster_base_folder = 'data/climate_data/climate_processed_data/monthly_longterm_wc2.1_2.5min_clipped_split_folder_setnodata'
raster_base_folder ='data/climate_data/climate_processed_data/new_gee_selected_months_era5land_clipped_split_folder'
anno_ym_mergedpresthinned = ae.annotate_points_by_time(points_gdf, date_column, raster_base_folder, date_format="%Y-%m")
anno_ym_mergedpresthinned['class'] = 1
display(anno_ym_mergedpresthinned.head())


with open("maxent_output_nov/maxent_output_nov_27/anno_ym_3kmthinned.pkl", "wb") as f:
    pickle.dump(anno_ym_mergedpresthinned, f)

# Do the same with 5km-thinned, 3km-thinned

## prepare late_df_gdf annotated by montly (wc) or yearmonthly (era5)

In [ ]:
#@title prepare late_df_gdf annotated by montly (wc) or yearmonthly (era5)

#montly (wc) => anno_m_late_df_gdf => anno_m_late_df_gdf_cleaned
points_gdf = late_df_gdf
date_column = 'Date'
raster_base_folder = 'data/climate_data/climate_processed_data/monthly_longterm_wc2.1_2.5min_clipped_split_folder_setnodata'
anno_m_late_df_gdf = ae.annotate_points_by_time(points_gdf, date_column, raster_base_folder, date_format="%m")
display(anno_m_late_df_gdf.head())

#2. clean the column name to be a base variable name
annotated_dfs = []
for time_key, group in anno_m_late_df_gdf.groupby("time_key"):
    # Drop columns with any NA values in the current group
    group_cleaned = group.dropna(axis=1)
    # Clean column names
    cleaned_columns = {}
    for col in group_cleaned.columns:
        if col.startswith('wc2.1_30s_'):
            # print('From old col:',col)
            parts = col.split('_')
            # Remove 'wc2.1_30s_' and the last part (month number)
            new_col = '_'.join(parts[2:-1])
            print('From old col:',col,'To new col:',new_col)
            cleaned_columns[col] = new_col
        else:
            cleaned_columns[col] = col # Keep other columns as they are
    print(f'cleaned_columns for {time_key}', cleaned_columns)
    group_cleaned = group_cleaned.rename(columns=cleaned_columns)
    annotated_dfs.append(group_cleaned)
    print('current annotated_dfs length:', len(annotated_dfs))
# Concatenate the cleaned dataframes
anno_m_late_df_gdf_cleaned = pd.concat(annotated_dfs, ignore_index=True)
# 3. add class to set it as presence points
anno_m_late_df_gdf_cleaned['class'] = 1
display(anno_m_late_df_gdf_cleaned.head())
display(anno_m_late_df_gdf_cleaned.columns)

#yearmonthly (era5) => anno_ym_late_df_gdf
points_gdf = late_df_gdf
date_column = 'Date'
# raster_base_folder = 'data/climate_data/climate_processed_data/monthly_longterm_wc2.1_2.5min_clipped_split_folder_setnodata'
raster_base_folder ='data/climate_data/climate_processed_data/new_gee_selected_months_era5land_clipped_split_folder'
anno_ym_late_df_gdf = ae.annotate_points_by_time(points_gdf, date_column, raster_base_folder, date_format="%Y-%m")
anno_ym_late_df_gdf['class'] = 1
display(anno_ym_late_df_gdf.head())

with open("maxent_output_nov/maxent_output_nov_27/anno_m_late_df_gdf_cleaned.pkl", "wb") as f:
    pickle.dump(anno_m_late_df_gdf_cleaned, f)
with open("maxent_output_nov/maxent_output_nov_27/anno_ym_late_df_gdf.pkl", "wb") as f:
    pickle.dump(anno_ym_late_df_gdf, f)

# Run MaxEnt for all combinations

- three presence data sets: thinned9km, thinned5km, thinned3km  
- three background data sets: bg1uniform, bg2lai, bg3MeanA3yr
- five climate data sets: set 1 era5land, set2 era5yearmonthly, set3 wcbioclim, set4 wcmonthly, set5 wcmonthlybg
- three checkerboad grid sizes: 0.5 degree, 0.4 degree, 0.3 degree


The examples here are for when we used grid size =  0.3 degree for 3 x 3 x 5 = 45 input combinations. But it will be the same for Grid size = 0.4 (another 45 models) and 0.5 (the last 45 models), and we only need to change
**grid_size = 0.3** to **0.4** and **0.5** in the line below:

- currentout = **maxent_single**(presence,background,rasters, labels, name ,
                                 bound, kmd_zone, outputpath = outputpath, **grid_size = 0.3**,
                                 bar = False, future_rasters = None, late_df_gdf = late_df_gdf)

or

- currentout = **maxent_single_timeaware**(annotated,rasters, labels, name, bound,kmd_zone,
                                       outputpath = outputpath, **grid_size = 0.3**, beta_multiplier = 1.5,
                                       bar = False, future_rasters = None,feature_types = 'lpqh',
                                       anno_late_df_gdf = anno_m_late_df_gdf_cleaned)

## 3 climate - thinned9km thinned5km thinned3km

In [ ]:
#@title 3 climate - thinned9km thinned5km thinned3km
presence = preschoice_dict['thinned9km']
rasterdictlist = [raster_dict_bioclim, raster_dict_era5land_relabelled, raster_dict_monthlybg_relabelled]
namerasterlist = ['wcbioclim','era5land','wcmonthlybg']
namepresence = 'thinned9km'
listofoutput3x3 = []
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 3 climate grid_size = 0.3'
start_time = time.time()
for i, (key, value) in enumerate(bgchoice_dict.items()):
    background = value
    namebg = key
    for j, raster_dict in enumerate(rasterdictlist):
      rasters = [value for key, value in raster_dict.items() if "leaf" not in key]
      labels = [key for key, value in raster_dict.items() if "leaf" not in key]
      nameraster = namerasterlist[j]
      name = [namepresence,namebg,nameraster]
      bound = landscape_gdf
      kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
      currentout = ae.maxent_single(presence,background,rasters, labels, name ,
                                 bound, kmd_zone, outputpath = outputpath, grid_size = 0.3,
                                 bar = False, future_rasters = None, late_df_gdf = late_df_gdf)

      listofoutput3x3.append(currentout)
      end_time = time.time()
      print(f" ////// FIN {namepresence} x {i} {namebg} x {j} {nameraster} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////")

with open(f"{outputpath}/{namepresence}_listofoutput3x3.pkl", "wb") as f:
    pickle.dump(listofoutput3x3, f)

presence = preschoice_dict['thinned5km']
rasterdictlist = [raster_dict_bioclim, raster_dict_era5land_relabelled, raster_dict_monthlybg_relabelled]
namerasterlist = ['wcbioclim','era5land','wcmonthlybg']
namepresence = 'thinned5km'
listofoutput3x3 = []
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 3 climate grid_size = 0.3'
start_time = time.time()
for i, (key, value) in enumerate(bgchoice_dict.items()):
    background = value
    namebg = key
    for j, raster_dict in enumerate(rasterdictlist):
      rasters = [value for key, value in raster_dict.items() if "leaf" not in key]
      labels = [key for key, value in raster_dict.items() if "leaf" not in key]
      nameraster = namerasterlist[j]
      name = [namepresence,namebg,nameraster]
      bound = landscape_gdf
      kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
      currentout = ae.maxent_single(presence,background,rasters, labels, name ,
                                 bound, kmd_zone, outputpath = outputpath, grid_size = 0.3,
                                 bar = False, future_rasters = None, late_df_gdf = late_df_gdf)

      listofoutput3x3.append(currentout)
      end_time = time.time()
      print(f" ////// FIN {namepresence} x {i} {namebg} x {j} {nameraster} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////")

with open(f"{outputpath}/{namepresence}_listofoutput3x3.pkl", "wb") as f:
    pickle.dump(listofoutput3x3, f)


presence = preschoice_dict['thinned3km']
rasterdictlist = [raster_dict_bioclim, raster_dict_era5land_relabelled, raster_dict_monthlybg_relabelled]
namerasterlist = ['wcbioclim','era5land','wcmonthlybg']
namepresence = 'thinned3km'
listofoutput3x3 = []
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 3 climate grid_size = 0.3'
start_time = time.time()
for i, (key, value) in enumerate(bgchoice_dict.items()):
    background = value
    namebg = key
    for j, raster_dict in enumerate(rasterdictlist):
      rasters = [value for key, value in raster_dict.items() if "leaf" not in key]
      labels = [key for key, value in raster_dict.items() if "leaf" not in key]
      nameraster = namerasterlist[j]
      name = [namepresence,namebg,nameraster]
      bound = landscape_gdf
      kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
      currentout = ae.maxent_single(presence,background,rasters, labels, name ,
                                 bound, kmd_zone, outputpath = outputpath, grid_size = 0.3,
                                 bar = False, future_rasters = None, late_df_gdf = late_df_gdf)

      listofoutput3x3.append(currentout)
      end_time = time.time()
      print(f" ////// FIN {namepresence} x {i} {namebg} x {j} {nameraster} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////")

with open(f"{outputpath}/{namepresence}_listofoutput3x3.pkl", "wb") as f:
    pickle.dump(listofoutput3x3, f)

## wcmonthly - thinned9km thinned5km thinned3km

In [ ]:
#@title wcmonthly - thinned9km thinned5km thinned3km

# with open("maxent_output_nov/maxent_output_nov_20 cvtrain 1 climate/anno_m_9kmthinned_cleaned.pkl", "rb") as f:
#     anno_m_9kmthinned_cleaned = pickle.load(f)
# with open("maxent_output_nov/maxent_output_nov_20 cvtrain 1 climate/anno_m_5kmthinned_cleaned.pkl", "rb") as f:
#     anno_m_5kmthinned_cleaned = pickle.load(f)
# with open("maxent_output_nov/maxent_output_nov_20 cvtrain 1 climate/anno_m_3kmthinned_cleaned.pkl", "rb") as f:
#     anno_m_3kmthinned_cleaned = pickle.load(f)

listofoutput3x1wcmonthly = []
rasters = [value for key, value in raster_dict_monthlybg_relabelled.items() if "leaf" not in key]
labels = [key for key, value in raster_dict_monthlybg_relabelled.items() if "leaf" not in key]
data_pres = anno_m_9kmthinned_cleaned[['geometry', 'class'] + labels]
namepresence = 'thinned9km'

nameraster = 'wcmonthly' #don't confuse with wcmonthlybg where presence also annotated by raster_dict_monthlybg_relabelled
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 2 climate grid_size = 0.3'
bound = landscape_gdf
kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
start_time = time.time()
for i, (key,bg) in enumerate(bgchoice_dict.items()):
  print('start annotating', key)
  annobase_m_bg = ela.annotate(bg, rasters, labels, drop_na=True, quiet=False) # some gen bg points may be outside the landscape, so drop_na=True
  annobase_m_bg['class'] = 0
  data_back = annobase_m_bg[annobase_m_bg['class'] == 0][['geometry', 'class'] + labels]
  data_manual = pd.concat([data_pres,data_back], axis = 0)
  namebg = f'{key}'
  name = [namepresence,namebg,nameraster]
  print(name)
  annotated = data_manual
  currentout = ae.maxent_single_timeaware(annotated,rasters, labels, name, bound,kmd_zone,
                                       outputpath = outputpath,grid_size = 0.3, beta_multiplier = 1.5,
                                       bar = False, future_rasters = None,feature_types = 'lpqh',
                                       anno_late_df_gdf = anno_m_late_df_gdf_cleaned)
  listofoutput3x1wcmonthly.append(currentout)
  end_time = time.time()
  print(f"FIN {i}////// {namebg} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////\n")

with open(f"{outputpath}/{namepresence}listofoutput3x1wcmonthly.pkl", "wb") as f:
    pickle.dump(listofoutput3x1wcmonthly, f)

listofoutput3x1wcmonthly = []
rasters = [value for key, value in raster_dict_monthlybg_relabelled.items() if "leaf" not in key]
labels = [key for key, value in raster_dict_monthlybg_relabelled.items() if "leaf" not in key]
data_pres = anno_m_5kmthinned_cleaned[['geometry', 'class'] + labels]
namepresence = 'thinned5km'

nameraster = 'wcmonthly' #don't confuse with wcmonthlybg where presence also annotated by raster_dict_monthlybg_relabelled
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 2 climate grid_size = 0.3'
bound = landscape_gdf
kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
start_time = time.time()
for i, (key,bg) in enumerate(bgchoice_dict.items()):
  print('start annotating', key)
  annobase_m_bg = ela.annotate(bg, rasters, labels, drop_na=True, quiet=False) # some gen bg points may be outside the landscape, so drop_na=True
  annobase_m_bg['class'] = 0
  data_back = annobase_m_bg[annobase_m_bg['class'] == 0][['geometry', 'class'] + labels]
  data_manual = pd.concat([data_pres,data_back], axis = 0)
  namebg = f'{key}'
  name = [namepresence,namebg,nameraster]
  print(name)
  annotated = data_manual
  currentout = ae.maxent_single_timeaware(annotated,rasters, labels, name, bound,kmd_zone,
                                       outputpath = outputpath,grid_size = 0.3, beta_multiplier = 1.5,
                                       bar = False, future_rasters = None,feature_types = 'lpqh',
                                       anno_late_df_gdf = anno_m_late_df_gdf_cleaned)
  listofoutput3x1wcmonthly.append(currentout)
  end_time = time.time()
  print(f"FIN {i}////// {namebg} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////\n")
with open(f"{outputpath}/{namepresence}listofoutput3x1wcmonthly.pkl", "wb") as f:
    pickle.dump(listofoutput3x1wcmonthly, f)

listofoutput3x1wcmonthly = []
rasters = [value for key, value in raster_dict_monthlybg_relabelled.items() if "leaf" not in key]
labels = [key for key, value in raster_dict_monthlybg_relabelled.items() if "leaf" not in key]
data_pres = anno_m_3kmthinned_cleaned[['geometry', 'class'] + labels]
namepresence = 'thinned3km'

nameraster = 'wcmonthly' #don't confuse with wcmonthlybg where presence also annotated by raster_dict_monthlybg_relabelled
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 2 climate grid_size = 0.3'
bound = landscape_gdf
kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
start_time = time.time()
for i, (key,bg) in enumerate(bgchoice_dict.items()):
  print('start annotating', key)
  annobase_m_bg = ela.annotate(bg, rasters, labels, drop_na=True, quiet=False) # some gen bg points may be outside the landscape, so drop_na=True
  annobase_m_bg['class'] = 0
  data_back = annobase_m_bg[annobase_m_bg['class'] == 0][['geometry', 'class'] + labels]
  data_manual = pd.concat([data_pres,data_back], axis = 0)
  namebg = f'{key}'
  name = [namepresence,namebg,nameraster]
  print(name)
  annotated = data_manual
  currentout = ae.maxent_single_timeaware(annotated,rasters, labels, name, bound,kmd_zone,
                                       outputpath = outputpath,grid_size = 0.3, beta_multiplier = 1.5,
                                       bar = False, future_rasters = None,feature_types = 'lpqh',
                                       anno_late_df_gdf = anno_m_late_df_gdf_cleaned)
  listofoutput3x1wcmonthly.append(currentout)
  end_time = time.time()
  print(f"FIN {i}////// {namebg} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////\n")
with open(f"{outputpath}/{namepresence}listofoutput3x1wcmonthly.pkl", "wb") as f:
    pickle.dump(listofoutput3x1wcmonthly, f)

## era5yearmonthly - thinned9km thinned5km thinned3km

In [ ]:
#@title era5yearmonthly - thinned9km thinned5km thinned3km

listofoutput3x1era5yearmonthly = []
rasters = [value for key, value in raster_dict_era5land_relabelled.items() if "leaf" not in key]
labels = [key for key, value in raster_dict_era5land_relabelled.items() if "leaf" not in key]
data_pres = anno_ym_9kmthinned[['geometry', 'class'] + labels]
namepresence = 'thinned9km'

nameraster = 'era5yearmonthly' #don't confuse with wcmonthlybg where presence also annotated by raster_dict_monthlybg_relabelled
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 2 climate grid_size = 0.3'
bound = landscape_gdf
kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
start_time = time.time()
for i, (key,bg) in enumerate(bgchoice_dict.items()):
  print('start annotating', key)
  annobase_m_bg = ela.annotate(bg, rasters, labels, drop_na=True, quiet=False) # some gen bg points may be outside the landscape, so drop_na=True
  annobase_m_bg['class'] = 0
  data_back = annobase_m_bg[annobase_m_bg['class'] == 0][['geometry', 'class'] + labels]
  data_manual = pd.concat([data_pres,data_back], axis = 0)
  namebg = f'{key}'
  name = [namepresence,namebg,nameraster]
  print(name)
  annotated = data_manual
  currentout = ae.maxent_single_timeaware(annotated,rasters, labels, name, bound,kmd_zone,
                                       outputpath = outputpath,grid_size = 0.3, beta_multiplier = 1.5,
                                       bar = False, future_rasters = None,feature_types = 'lpqh',
                                       anno_late_df_gdf = anno_ym_late_df_gdf)
  listofoutput3x1era5yearmonthly.append(currentout)
  end_time = time.time()
  print(f"FIN {i}////// {namebg} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////\n")
with open(f"{outputpath}/{namepresence}listofoutput3x1era5yearmonthly.pkl", "wb") as f:
    pickle.dump(listofoutput3x1era5yearmonthly, f)
listofoutput3x1era5yearmonthly = []
rasters = [value for key, value in raster_dict_era5land_relabelled.items() if "leaf" not in key]
labels = [key for key, value in raster_dict_era5land_relabelled.items() if "leaf" not in key]
data_pres = anno_ym_5kmthinned[['geometry', 'class'] + labels]
namepresence = 'thinned5km'

nameraster = 'era5yearmonthly' #don't confuse with wcmonthlybg where presence also annotated by raster_dict_monthlybg_relabelled
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 2 climate grid_size = 0.3'
bound = landscape_gdf
kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
start_time = time.time()
for i, (key,bg) in enumerate(bgchoice_dict.items()):
  print('start annotating', key)
  annobase_m_bg = ela.annotate(bg, rasters, labels, drop_na=True, quiet=False) # some gen bg points may be outside the landscape, so drop_na=True
  annobase_m_bg['class'] = 0
  data_back = annobase_m_bg[annobase_m_bg['class'] == 0][['geometry', 'class'] + labels]
  data_manual = pd.concat([data_pres,data_back], axis = 0)
  namebg = f'{key}'
  name = [namepresence,namebg,nameraster]
  print(name)
  annotated = data_manual
  currentout = ae.maxent_single_timeaware(annotated,rasters, labels, name, bound,kmd_zone,
                                       outputpath = outputpath,grid_size = 0.3, beta_multiplier = 1.5,
                                       bar = False, future_rasters = None,feature_types = 'lpqh',
                                       anno_late_df_gdf = anno_ym_late_df_gdf)
  listofoutput3x1era5yearmonthly.append(currentout)
  end_time = time.time()
  print(f"FIN {i}////// {namebg} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////\n")

with open(f"{outputpath}/{namepresence}listofoutput3x1era5yearmonthly.pkl", "wb") as f:
    pickle.dump(listofoutput3x1era5yearmonthly, f)

listofoutput3x1era5yearmonthly = []
rasters = [value for key, value in raster_dict_era5land_relabelled.items() if "leaf" not in key]
labels = [key for key, value in raster_dict_era5land_relabelled.items() if "leaf" not in key]
data_pres = anno_ym_3kmthinned[['geometry', 'class'] + labels]
namepresence = 'thinned3km'

nameraster = 'era5yearmonthly' #don't confuse with wcmonthlybg where presence also annotated by raster_dict_monthlybg_relabelled
outputpath = 'maxent_output_nov/maxent_output_nov_27 checkerboardtrain 2 climate grid_size = 0.3'
bound = landscape_gdf
kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
start_time = time.time()
for i, (key,bg) in enumerate(bgchoice_dict.items()):
  print('start annotating', key)
  annobase_m_bg = ela.annotate(bg, rasters, labels, drop_na=True, quiet=False) # some gen bg points may be outside the landscape, so drop_na=True
  annobase_m_bg['class'] = 0
  data_back = annobase_m_bg[annobase_m_bg['class'] == 0][['geometry', 'class'] + labels]
  data_manual = pd.concat([data_pres,data_back], axis = 0)
  namebg = f'{key}'
  name = [namepresence,namebg,nameraster]
  print(name)
  annotated = data_manual
  currentout = ae.maxent_single_timeaware(annotated,rasters, labels, name, bound,kmd_zone,
                                       outputpath = outputpath,grid_size = 0.3, beta_multiplier = 1.5,
                                       bar = False, future_rasters = None,feature_types = 'lpqh',
                                       anno_late_df_gdf = anno_ym_late_df_gdf)
  listofoutput3x1era5yearmonthly.append(currentout)
  end_time = time.time()
  print(f"FIN {i}////// {namebg} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////\n")

with open(f"{outputpath}/{namepresence}listofoutput3x1era5yearmonthly.pkl", "wb") as f:
    pickle.dump(listofoutput3x1era5yearmonthly, f)

# Future prediction

## preprocess future data 

We already provided precessed future data in "1_data/climate_data/processed_future/" and therefore this part can be skipped

In [ ]:
#@title preprocess

raster_bioclim_future_ssp126 = ae.load_climatetif_to_dict_rec(climate_folder_path = '1_data/climate_data/raw_future/ssp126')
raster_bioclim_future_ssp370 = ae.load_climatetif_to_dict_rec(climate_folder_path = '1_data/climate_data/raw_future/ssp370')

# for key,value in raster_bioclim_future_ssp370.items():
#   pc.check_raster_properties(value)


# ori_folder_path = 'data/climate_data/climate_future_raw/ssp126'
# save_to_folder='data/climate_data/climate_future_processed/ssp126_setnodata'
# for tif in glob(os.path.join(f'{ori_folder_path}/*.tif')):
#   print(tif)
#   vc.visualise_rasters(tif)
#   pc.set_nodata_value_in_geotiff(tif, nodata_value=-9999, save_to_folder = save_to_folder)

# raster_bioclim_future_5_snd = ae.load_climatetif_to_dict_rec(climate_folder_path = save_to_folder)

# for key,value in raster_bioclim_future_5_snd.items():
#   pc.check_raster_properties(value)
#   vc.visualise_rasters(value)

def split_geotiff_bands(input_file, output_folder, prefix='bio'):
    """Split multi-band GeoTIFF into separate files."""
    os.makedirs(output_folder, exist_ok=True)

    with rasterio.open(input_file) as src:
        meta = src.meta.copy()
        meta['count'] = 1

        for band in range(1, src.count + 1):
            output_path = os.path.join(output_folder, f"{prefix}_{band:02d}.tif")
            with rasterio.open(output_path, 'w', **meta) as dst:
                dst.write(src.read(band), 1)
            print(f"Created: {os.path.basename(output_path)}")

    return [os.path.join(output_folder, f"{prefix}_{i:02d}.tif")
            for i in range(1, src.count + 1)]

In [ ]:
base_folder_forall = '1_data/climate_data/processed_future'
# raster_bioclim_future_ssp370 and raster_bioclim_future_ssp126 done
for key,value in raster_bioclim_future_ssp370.items():
  # pc.check_raster_properties(value)
  print(value)
  basename = os.path.basename(value).replace('_clipped.tif', '').replace('wc_30s_')
  # print(basename)
  pathway = value.split('/')[-2]
  save_folder_name = f'{base_folder_forall}/{basename}'
  print('=> save split band to', save_folder_name)
  splitfiles = split_geotiff_bands(value, save_folder_name, prefix='bio')

In [ ]:
# @title rename files once
bio_variable_definitions = {
    'bio_01':  "annual_mean_temp",
    'bio_02':  "mean_diurnal_range",
    'bio_03':  "isothermality",
    'bio_04':  "temp_seasonality",
    'bio_05':  "max_temp_warmest_month",
    'bio_06':  "min_temp_coldest_month",
    'bio_07':  "temp_annual_range",
    'bio_08':  "mean_temp_wettest_quarter",
    'bio_09':  "mean_temp_driest_quarter",
    'bio_10': "mean_temp_warmest_quarter",
    'bio_11': "mean_temp_coldest_quarter",
    'bio_12': "annual_prec",
    'bio_13': "prec_wettest_month",
    'bio_14': "prec_driest_month",
    'bio_15': "prec_seasonality",
    'bio_16': "prec_wettest_quarter",
    'bio_17': "prec_driest_quarter",
    'bio_18': "prec_warmest_quarter",
    'bio_19': "prec_coldest_quarter"
}


def rename_bio_keys(old_path):
  pattern = f'{old_path}/*.tif'
  raster_files = glob(pattern)
  raster_files
  import re
  for old_path in raster_files:
      dirname = os.path.dirname(old_path)
      basename = os.path.basename(old_path)
      # Extract the bio number (e.g., 'bio_1', 'bio_12')
      basename_no_ext = os.path.splitext(basename)[0]
      label = f"{basename_no_ext.split('_')[0]}_{basename_no_ext.split('_')[1]}"
      # if len(label.split('_')[-1]) < 2:
      #   label = label.replace('bio_', 'bio_0')
      # else:
      #   label = label
      print('label:', label)
      # match = re.match(r'(bio_\d+)', basename)
      match = re.match(r'(bio_\d+)', label)

      if match:
          bio_key = match.group(1)
          if bio_key in bio_variable_definitions:
              new_name = f"{label}_{bio_variable_definitions[bio_key]}.tif" # Assuming the file extension is .tif
              new_path = os.path.join(dirname, new_name)
              print('new_path:', new_path)
              try:
                  os.rename(old_path, new_path)
                  print(f"Renamed '{basename}' to '{new_name}'")
              except OSError as e:
                  print(f"Error renaming '{basename}': {e}")
          else:
              print(f"Warning: Key '{bio_key}' not found in bio_variable_definitions for file '{basename}'. Skipping rename.")
      else:
          print(f"Warning: Could not extract 'bio_XX' pattern from filename '{basename}'. Skipping rename.")

  print("\nFile renaming complete.")

  # Verify the renaming (optional)
  print("\nFiles in the directory after renaming:")
  !ls {old_path}

In [ ]:

for pathway in ['ssp126','ssp370']:
  for gcm in ['ACCESSCM2','BCCCSM2MR','MIROC6','MPIESM12HR','UKESM10LL']:
    name = f'1_data/climate_data/processed_future/{pathway}_{gcm}'
    print(name)
    rename_bio_keys(name)

In [ ]:
#@title Load future climate
# raster_bioclim_future = ae.load_climatetif_to_dict_rec(climate_folder_path = 'data/climate_data/climate_processed_data_future/future_wc_bio_clipped_tif_setnodata')
# raster_bioclim_future_ssp370_2040 = ae.load_climatetif_to_dict_rec(climate_folder_path = 'data/climate_data/climate_processed_data_future/future_wc_bio_ssp370_2040_clipped_setnodata')
# raster_bioclim_future_ssp126_2040 = ae.load_climatetif_to_dict_rec(climate_folder_path = 'data/climate_data/climate_processed_data_future/future_wc_bio_ssp126_2040_clipped_setnodata')


## Load model produced using ras wc bioclim


In [ ]:
#@title Load model produced using ras wc bioclim
#  1.Load model that use bioclim
# model_raswcbioclim_list = []
model_raswcbioclim_split_list = []
base_dir = 'path/to/wcbioclim_models/' #where you save the trained wcbioclime models
all_subdirs = sorted(glob(os.path.join(base_dir, 'ModelOut*')))
kw = '_thinned9km_bg2lai_wcbioclim'
matched_model = [s for s in all_subdirs if kw in s]
matched_model[0]
modelsplitela = glob(os.path.join(matched_model[0], '*_modelsplit.ela'))
modelsplitela
# for i, kw2 in enumerate(list(bgchoice_dict.keys())):
#   matched_bg = [s for s in matched_clim if kw2 in s]
#   print(i,kw2)
#   print(matched_bg)
#   # modelela = glob(os.path.join(matched_bg[0], '*_model.ela'))
#   modelsplitela = glob(os.path.join(matched_bg[0], '*_modelsplit.ela'))
#   # print(modelela)
#   print(modelsplitela)
#   # model_raswcbioclim_list.append(ela.load_object(modelela[0]))
model_raswcbioclim_split_list.append(ela.load_object(modelsplitela[0]))
model_raswcbioclim_split_list

## create function def maxent_projecting_future 

## run model in a loop

In [ ]:
#@title Use in a loop
#  1.Load model that use bioclim
namerasterdict = {}
for pathway in ['ssp126','ssp370']:
  for gcm in ['ACCESSCM2','BCCCSM2MR','MIROC6','MPIESM12HR','UKESM10LL']:
    name = f'1_data/climate_data/processed_future/{pathway}_{gcm}'
    print(name)
    basename = os.path.basename(name)
    shortname = basename #.replace('wc_30s_','').replace('_sepband','')
    namerasterdict[name] = shortname
namerasterdict

model_raswcbioclim_split_list[0]
# 2. apply model to raster

# test name = 'data/climate_data/climate_future_processed/wc_30s_ssp126_ACCESSCM2_sepband'
# raster_bioclim_future_ssp126_current = ae.load_climatetif_to_dict_rec(name)
# nameraster_current = namerasterdict[name]
# nameraster_current

rasterdictlist = [raster_dict_bioclim] #just for setting the right order of the lables. the real one used is future climate!
# namerasterlist = ['wcbioclimf_ssp370_40']

namepresence = 'thinned9km'
listofoutputx10future = []
output_path = 'maxent_output_nov_future'
start_time = time.time()
namebg = 'bg2lai'
bound = landscape_gdf
kmd_zone = augment_cleaned_counties_gdf[['County', 'geometry']]
background = bgchoice_dict[namebg]
for i, (filepath,shortname) in enumerate(namerasterdict.items()):
    nameraster_current = shortname
    print('Start', i, shortname)
    raster_bioclim_future_ssp126_current = ae.load_climatetif_to_dict_rec(filepath)
    # model = model_raswcbioclim_list[i]
    modelsplit = model_raswcbioclim_split_list[0]
    rasters = [value for key, value in rasterdictlist[0].items() if "leaf" not in key]
    labels = [key for key, value in rasterdictlist[0].items() if "leaf" not in key]
    future_rasters = [raster_bioclim_future_ssp126_current[k] for k in labels]
    print(labels)
    print(future_rasters)
    nameraster = nameraster_current
    name = [namepresence,namebg,nameraster]
    output_model_dir = f'{output_path}/ModelOut_{namepresence}_{namebg}_{nameraster}'
    os.makedirs(output_model_dir, exist_ok=True)
    currentoutfuture = ae.maxent_projecting_future(modelsplit, future_rasters, labels, output_model_dir, name,kmd_zone, zone_var = 'County')
    listofoutputx10future.append(currentoutfuture)

    # saving manually (not automatic like when run maxent_single())
    map, zonal = currentoutfuture
    print(f"Using future_rastersfor visualising predictions on map, must contain the bands:\n {labels}")
    fig_modelsplitmap_f, output_raster_split_f = map
    zs_f, fig_zonal_bar_f, figpredsplit_f, fig_zonal_bar_f_sorted = zonal
    fig_modelsplitmap_f.savefig(f"{output_model_dir}/{namepresence}_{namebg}_{nameraster}_future_modelsplit_map.tiff", dpi=300)
    print("Out_future - Saved future habitat suitability map for checkerboard model\n")
    zs_f.to_csv(f"{output_model_dir}/{namepresence}_{namebg}_{nameraster}_future_zonal_stats.csv")
    print("Out_future - Saved zonal stat csv\n")
    fig_zonal_bar_f.savefig(f"{output_model_dir}/{namepresence}_{namebg}_{nameraster}_future_zonal_stats.png", dpi=300,bbox_inches='tight')
    fig_zonal_bar_f_sorted.savefig(f"{output_model_dir}/{namepresence}_{namebg}_{nameraster}_future_zonal_sorted.png", dpi=300,bbox_inches='tight')
    print("Out_future - Saved zonal stat bar graph and sorted\n")
    figpredsplit_f.savefig(f"{output_model_dir}/{namepresence}_{namebg}_{nameraster}_future_zonal_predsplit.tiff", dpi=300,bbox_inches='tight')
    print("Out_future - Saved zonal pred and predsplit images\n")
    end_time = time.time()
    print(f"FIN ////// {i} x {nameraster} ////// Elapsed time since first: {end_time - start_time:.2f} seconds //////\n")



with open("maxent_output_nov_future/listofoutputx10future.pkl", "wb") as f:
    pickle.dump(listofoutputx10future, f)